# Notebook 05 — Materials Builder
**Mission Veteran AI · Veteran Career Navigator**

This is the key notebook for the evidence-bounded generation pipeline:

```
tblPrompts (PT-002, PT-005)
       ↓
Evidence-bounded prompt assembly
       ↓
IBM Granite (watsonx.ai)
       ↓
Parse output → tag every sentence (EVIDENCED / INFERRED / MISSING)
       ↓
Block INFERRED and MISSING from export
       ↓
Gate hold report
```

Produces both a resume draft (PT-002) and a cover letter draft (PT-005).

**Prerequisites**
- Notebook 04 completed (resume draft inputs available)  
- `.env` with `WATSONX_API_KEY`, `WATSONX_URL`, `WATSONX_PROJECT_ID`

In [ ]:
%pip install -q ibm-watsonx-ai python-dotenv

In [ ]:
import json, os, re, sys
from dotenv import load_dotenv

load_dotenv()
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

API_KEY    = os.environ['WATSONX_API_KEY']
WX_URL     = os.environ.get('WATSONX_URL', 'https://us-south.ml.cloud.ibm.com')
PROJECT_ID = os.environ['WATSONX_PROJECT_ID']
print('Credentials loaded ✓')

## 1 · Load PT-002 (resume) and PT-005 (cover letter) from tblPrompts

In [ ]:
KB_PATH = os.path.join(ROOT, 'knowledge-bases', 'prompt_templates.json')
with open(KB_PATH, 'r', encoding='utf-8') as f:
    templates = json.load(f)

pt002 = next(t for t in templates if t['name'] == 'resume_draft_generation')
pt005 = next(t for t in templates if t['name'] == 'cover_letter_generation')
print(f"PT-002: {pt002['template_id']} ✓")
print(f"PT-005: {pt005['template_id']} ✓")

## 2 · Inputs (from Notebook 04 or real pipeline outputs)

In [ ]:
SERVICE_BRANCH   = 'Army'
SPECIALTY_CODE   = '92A'
TARGET_ROLE      = 'Supply Chain Operations Manager'
COMPANY_NAME     = 'Acme Logistics Inc.'   # for cover letter

SERVICE_SUMMARY = (
    'Managed automated supply logistics for a 500-person organization over 4 years, '
    'overseeing a $2.1M annual inventory. Led a 12-member team, improved on-time '
    'delivery from 82% to 96%, and implemented a cycle-count program that reduced '
    'shrinkage by 18%.'
)

TRANSLATED_BULLETS = [
    'Directed a 12-member logistics team, improving on-time delivery rates from 82% to 96% over 18 months.',
    'Managed a $2.1M annual inventory across multiple warehouse locations, achieving a 98.7% accountability rate.',
    'Implemented a cycle-count reconciliation program that reduced inventory shrinkage by 18%.',
]

SKILL_INVENTORY = [
    {'skill_name': 'logistics coordination',        'category': 'Operations',    'confidence': 'High'},
    {'skill_name': 'team leadership',               'category': 'Leadership',    'confidence': 'High'},
    {'skill_name': 'inventory management',          'category': 'Supply Chain',  'confidence': 'High'},
    {'skill_name': 'budget oversight',              'category': 'Finance',       'confidence': 'Medium'},
    {'skill_name': 'cross-functional coordination', 'category': 'Collaboration', 'confidence': 'High'},
]

REQUIRED_SKILLS = [
    'supply chain coordination', 'inventory management', 'vendor management',
    'budget oversight', 'ERP/WMS proficiency', 'cross-functional team leadership',
]

JOB_ANALYSIS = {
    'role_title': TARGET_ROLE,
    'seniority_level': 'Mid-level (3-5 years)',
    'required_skills': REQUIRED_SKILLS,
    'preferred_skills': ['APICS CSCP', 'Six Sigma', 'SAP'],
    'ats_keywords': ['supply chain', 'logistics', 'inventory', 'KPI', 'cross-functional'],
}
print('Inputs defined ✓')

## 3 · Helper: call Granite with a template

In [ ]:
from ibm_watsonx_ai import APIClient, Credentials
from ibm_watsonx_ai.foundation_models import ModelInference
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as Params

MODEL_ID = 'ibm/granite-4-h-small'
_client  = APIClient(credentials=Credentials(url=WX_URL, api_key=API_KEY))

def call_granite(template, user_prompt):
    p = template['model_params']
    model = ModelInference(
        model_id=MODEL_ID, api_client=_client, project_id=PROJECT_ID,
        params={
            Params.MAX_NEW_TOKENS:    p['max_new_tokens'],
            Params.MIN_NEW_TOKENS:    p['min_new_tokens'],
            Params.TEMPERATURE:       p['temperature'],
            Params.REPETITION_PENALTY:p['repetition_penalty'],
        },
    )
    resp = model.generate(
        prompt=f"System:\n{template['system_prompt']}\nUser:\n{user_prompt}\nAssistant:\n"
    )
    raw = resp['results'][0]['generated_text'].strip()
    raw = re.sub(r'^```[a-z]*\n?', '', raw)
    return re.sub(r'\n?```$', '', raw)

print('Helper defined ✓')

## 4 · Generate tagged resume draft (PT-002)

In [ ]:
bullets_text = '\n'.join(f'- {b}' for b in TRANSLATED_BULLETS)
skills_text  = '\n'.join(f"- {s['skill_name']} ({s['category']}, {s['confidence']})" for s in SKILL_INVENTORY)
job_text     = json.dumps(JOB_ANALYSIS, indent=2)

resume_prompt = pt002['user_prompt_template'].format(
    service_branch=SERVICE_BRANCH,
    specialty_code=SPECIALTY_CODE,
    service_summary=SERVICE_SUMMARY[:1000],
    translated_bullets=bullets_text[:800],
    skill_inventory=skills_text[:600],
    job_analysis=job_text[:600],
    target_role=TARGET_ROLE,
)

resume_raw   = call_granite(pt002, resume_prompt)
resume_draft = json.loads(resume_raw)
print('Resume draft generated ✓')

## 5 · Generate tagged cover letter draft (PT-005)

In [ ]:
cover_prompt = pt005['user_prompt_template'].format(
    target_role=TARGET_ROLE,
    company_name=COMPANY_NAME,
    service_summary=SERVICE_SUMMARY[:800],
    translated_bullets=bullets_text[:600],
    required_skills=', '.join(REQUIRED_SKILLS),
    skill_inventory=skills_text[:400],
)

cover_raw   = call_granite(pt005, cover_prompt)
cover_draft = json.loads(cover_raw)
print('Cover letter draft generated ✓')

## 6 · Tag every sentence and show the full output

In [ ]:
ICONS = {'EVIDENCED': '✓', 'INFERRED': '⚠', 'MISSING': '✗'}

def show_tagged(label, items):
    print(f'\n── {label} ──')
    if isinstance(items, dict):
        items = [items]
    for item in items:
        if isinstance(item, str):
            print(f'  • {item}')
        else:
            tag  = item.get('tag', '?').upper()
            icon = ICONS.get(tag, '?')
            print(f'  [{tag}] {icon}  {item.get("text", "")}')

print('═' * 60)
print('RESUME DRAFT')
summary = resume_draft.get('summary', {})
if isinstance(summary, str):
    summary = {'text': summary, 'tag': 'INFERRED'}
show_tagged('Summary', [summary])
show_tagged('Bullets', resume_draft.get('bullets', []))
show_tagged('Skills Section', resume_draft.get('skills_section', []))

print()
print('═' * 60)
print('COVER LETTER DRAFT')
for section in ('opening', 'evidence', 'closing'):
    show_tagged(section.capitalize(), cover_draft.get(section, []))

## 7 · Collect all blocked sentences and evaluate the gate

In [ ]:
def collect_blocked(tagged_items, source):
    blocked = []
    if isinstance(tagged_items, dict):
        tagged_items = [tagged_items]
    for item in tagged_items:
        if isinstance(item, dict) and item.get('tag', '').upper() in ('INFERRED', 'MISSING'):
            blocked.append({'text': item.get('text',''), 'tag': item['tag'], 'source': source})
    return blocked

all_blocked = []
# Resume
all_blocked += collect_blocked(summary, 'resume_summary')
all_blocked += collect_blocked(resume_draft.get('bullets', []), 'resume_bullet')
# Cover letter
for section in ('opening', 'evidence', 'closing'):
    all_blocked += collect_blocked(cover_draft.get(section, []), f'cover_letter_{section}')

inferred_n = sum(1 for b in all_blocked if b['tag'].upper() == 'INFERRED')
missing_n  = sum(1 for b in all_blocked if b['tag'].upper() == 'MISSING')
gate_status = 'OPEN' if not all_blocked else 'BLOCKED'

print(f'Blocked: {len(all_blocked)}  (INFERRED: {inferred_n}, MISSING: {missing_n})')
print(f'Gate status: {gate_status}')

if all_blocked:
    print()
    for b in all_blocked:
        print(f"  [{b['tag']}] ({b['source']}) {b['text']}")